# 🛡️ EDDS — Deepfake Detection Model Training
> **GitHub Repo:** https://github.com/ShivamSri-8/Deepafake-Defence-System  
> **Models:** EfficientNet-B4 · ResNet50 · Xception  
> **Run cells one by one from top to bottom ↓**

## ✅ Step 1 — Verify GPU is available

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')
else:
    print('⚠️  No GPU found. Go to Runtime → Change runtime type → GPU')

## ✅ Step 2 — Mount Google Drive (to store data & model weights)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted at /content/drive')

## ✅ Step 3 — Clone the GitHub Repo

In [ ]:
import os

REPO_URL = 'https://github.com/ShivamSri-8/Deepafake-Defence-System.git'
REPO_DIR = '/content/Deepafake-Defence-System'

if os.path.exists(REPO_DIR):
    print('Repo already cloned. Pulling latest changes...')
    os.system(f'git -C {REPO_DIR} pull')
else:
    os.system(f'git clone {REPO_URL} {REPO_DIR}')

os.chdir(REPO_DIR)
print('📁 Working directory:', os.getcwd())

## ✅ Step 4 — Install Dependencies

In [ ]:
!pip install -r requirements.txt -q
!pip install pretrainedmodels -q   # Required for Xception model
print('✅ All dependencies installed!')

## ✅ Step 5 — Set Up Dataset Folder

> 📌 **Your dataset must be inside Google Drive** in this layout:
> ```
> MyDrive/
> └── deepfake_dataset/
>     ├── train/
>     │   ├── real/    ← real face images
>     │   └── fake/    ← deepfake images
>     └── val/
>         ├── real/
>         └── fake/
> ```
> Change `DATASET_DRIVE_PATH` below to match where you put your data.

In [ ]:
import os

# ⬇️ CHANGE THIS to your actual dataset folder in Google Drive
DATASET_DRIVE_PATH = '/content/drive/MyDrive/deepfake_dataset'

# Symlink so the training script can find it as 'data/'
if not os.path.exists('data'):
    os.symlink(DATASET_DRIVE_PATH, 'data')
    print('✅ Dataset linked at ./data')
else:
    print('✅ ./data already exists')

# Quick sanity check
for split in ['train/real', 'train/fake', 'val/real', 'val/fake']:
    path = os.path.join('data', split)
    if os.path.exists(path):
        count = len(os.listdir(path))
        print(f'  [{split}] — {count} files')
    else:
        print(f'  ❌ MISSING: {path}  ← create this folder in Drive!')

## ✅ Step 6 — Create Output Folder for Weights on Drive

In [ ]:
import os

WEIGHTS_DRIVE = '/content/drive/MyDrive/deepfake_weights'
os.makedirs(WEIGHTS_DRIVE, exist_ok=True)

LOCAL_WEIGHTS = 'models/weights'
os.makedirs(LOCAL_WEIGHTS, exist_ok=True)

print('✅ Weight folders ready')
print('  Local  :', LOCAL_WEIGHTS)
print('  Drive  :', WEIGHTS_DRIVE)

## 🚀 Step 7 — Train the Model

> 🔧 **Tune these settings:**
> - `--arch`      → `efficientnet` | `resnet50` | `xception` | `all`
> - `--epochs`    → how many training rounds (start with 10 to test)
> - `--batch-size`→ 16 is good for Colab T4 GPU
> - `--lr`        → learning rate (0.0001 is a safe default)

In [ ]:
!python ai-engine/training/train_pytorch.py \
    --data-dir data \
    --arch efficientnet \
    --epochs 10 \
    --batch-size 16 \
    --lr 0.0001 \
    --grad-accum 2 \
    --log-every 50

## ✅ Step 8 — Copy Trained Weights to Google Drive (so they don't get lost)

In [ ]:
import shutil, os, glob

WEIGHTS_DRIVE = '/content/drive/MyDrive/deepfake_weights'
weight_files = glob.glob('models/weights/*.pt')

if weight_files:
    for f in weight_files:
        dest = os.path.join(WEIGHTS_DRIVE, os.path.basename(f))
        shutil.copy2(f, dest)
        print(f'  ✅ Copied: {os.path.basename(f)} → Drive')
else:
    print('⚠️  No weight files found. Did training complete successfully?')

## 🔁 Optional — Resume Training If Session Restarted

In [ ]:
# First copy checkpoint back from Drive
import shutil, glob, os

WEIGHTS_DRIVE = '/content/drive/MyDrive/deepfake_weights'
os.makedirs('models/weights', exist_ok=True)

for f in glob.glob(os.path.join(WEIGHTS_DRIVE, '*.pt')):
    dest = os.path.join('models/weights', os.path.basename(f))
    shutil.copy2(f, dest)
    print(f'  Restored: {os.path.basename(f)}')

# Then resume training
!python ai-engine/training/train_pytorch.py \
    --data-dir data \
    --arch efficientnet \
    --epochs 30 \
    --batch-size 16 \
    --lr 0.0001 \
    --resume